In [1]:
import boto3
import json
import re
import uuid
import os
import hashlib
from pydantic import BaseModel, Field
from dataclasses import dataclass
from typing import Optional
from datetime import datetime
from dotenv import load_dotenv
import mlflow
import warnings

# Load env file
load_dotenv()

os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("DAGSHUB_TOKEN")

# Tracking server
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("inframind_experiment")

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name="ap-south-1",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

x:\nasim_xhqpjmy\Code\GenAI\InfraMind\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Llama 3 Pricing on AWS Bedrock (On-Demand per 1k tokens)
PRICING = {
    "meta.llama3-8b-instruct-v1:0": {"input": 0.0003, "output": 0.0006},
    "meta.llama3-70b-instruct-v1:0": {"input": 0.00265, "output": 0.0035}
}

In [3]:
#Output Schema
class RCAOutput(BaseModel):
    incident_id:      str   = Field(description="Unique ID for the incident")
    severity:         str   = Field(description="Critical, High, Medium, or Low")
    summary:          str   = Field(description="One-sentence summary of what happened")
    root_cause:       str   = Field(description="The primary technical reason for the failure")
    immediate_fix:    str   = Field(description="What to do right now to restore service")
    confidence_score: float = Field(description="AI's confidence (0.0 to 1.0)")
    model_used:       str   = Field(description="Llama-3-8B or Llama-3-70B")

json_schema = RCAOutput.model_json_schema()

In [4]:
# LOG NORMALIZER
# Accepts: CloudWatch JSON, k8s syslog, Python/Java app logs,nginx/apache access logs, plain text error strings
@dataclass
class NormalizedLog:
    timestamp:     Optional[str]  # ISO string or None
    severity:      str            # ERROR / WARN / INFO / DEBUG
    service:       Optional[str]  # service / container name if detectable
    message:       str            # clean core error message
    raw:           str            # always preserve original
    source_format: str            # which parser matched


def normalize_log(raw_log: str) -> NormalizedLog:
    """
    Detects log format and normalizes into a consistent structure.
    Falls back gracefully — never crashes, always returns something useful.
    """
    raw_log = raw_log.strip()

    # ── 1. CloudWatch / structured JSON ──────────────────────────────────
    # {"timestamp":"...","level":"ERROR","message":"...","service":"api"}
    try:
        data = json.loads(raw_log)
        if isinstance(data, dict) and any(k in data for k in ("message", "msg", "@message")):
            msg = data.get("message") or data.get("msg") or data.get("@message", "")
            lvl = (
                data.get("level") or data.get("severity") or
                data.get("log_level") or data.get("levelname") or "ERROR"
            ).upper()
            ts = (
                data.get("timestamp") or data.get("time") or
                data.get("@timestamp") or data.get("eventTime")
            )
            svc = (
                data.get("service") or data.get("container") or
                data.get("source") or data.get("logger")
            )
            return NormalizedLog(
                timestamp=ts, severity=lvl,
                service=svc, message=msg,
                raw=raw_log, source_format="cloudwatch_json"
            )
    except (json.JSONDecodeError, TypeError):
        pass

    # ── 2. Kubernetes / kubelet syslog ────────────────────────────────────
    # E0115 10:23:45.123456    42 pod_workers.go:191] Error syncing pod
    k8s = re.match(
        r'^([EWID])(\d{4})\s+([\d:.]+)\s+\d+\s+[\w./]+:\d+\]\s+(.+)$',
        raw_log
    )
    if k8s:
        level_map = {"E": "ERROR", "W": "WARN", "I": "INFO", "D": "DEBUG"}
        return NormalizedLog(
            timestamp=k8s.group(2),
            severity=level_map.get(k8s.group(1), "ERROR"),
            service="kubelet",
            message=k8s.group(4),
            raw=raw_log, source_format="k8s_syslog"
        )

    # ── 3. Standard app log (Python / Java / Node) ───────────────────────
    # 2024-01-15 10:23:45 [ERROR] message
    # 2024-01-15T10:23:45.123Z ERROR  message
    # 2024/01/15 10:23:45 ERROR message
    app = re.match(
        r'^(\d{4}[-/]\d{2}[-/]\d{2}[T\s][\d:.]+Z?)\s+\[?(\w+)\]?\s+(.+)$',
        raw_log
    )
    if app:
        return NormalizedLog(
            timestamp=app.group(1),
            severity=app.group(2).upper(),
            service=None,
            message=app.group(3),
            raw=raw_log, source_format="standard_app"
        )

    # ── 4. Nginx / Apache access log ─────────────────────────────────────
    # 192.168.1.1 - - [15/Jan/2024:10:23:45 +0000] "GET /api/v1 HTTP/1.1" 500 1234
    nginx = re.match(
        r'^(\S+)\s+-\s+-\s+\[([^\]]+)\]\s+"([^"]+)"\s+(\d{3})\s+(\d+)',
        raw_log
    )
    if nginx:
        status = int(nginx.group(4))
        severity = "ERROR" if status >= 500 else "WARN" if status >= 400 else "INFO"
        return NormalizedLog(
            timestamp=nginx.group(2),
            severity=severity,
            service="nginx",
            message=f"HTTP {status} — {nginx.group(3)}",
            raw=raw_log, source_format="nginx_access"
        )

    # ── 5. Syslog format ─────────────────────────────────────────────────
    # Jan 15 10:23:45 hostname servicename[PID]: message
    syslog = re.match(
        r'^(\w{3}\s+\d+\s+[\d:]+)\s+(\S+)\s+([\w/-]+)\[\d+\]:\s+(.+)$',
        raw_log
    )
    if syslog:
        msg = syslog.group(4)
        severity = "ERROR" if re.search(r'error|fail|fatal|crit', msg, re.I) else \
                   "WARN"  if re.search(r'warn|warning',          msg, re.I) else "INFO"
        return NormalizedLog(
            timestamp=syslog.group(1),
            severity=severity,
            service=syslog.group(3),
            message=msg,
            raw=raw_log, source_format="syslog"
        )

    # ── 6. Docker / container log with stream prefix ─────────────────────
    # 2024-01-15T10:23:45.123456789Z stdout F actual log message
    docker = re.match(
        r'^(\d{4}-\d{2}-\d{2}T[\d:.]+Z)\s+\w+\s+\w\s+(.+)$',
        raw_log
    )
    if docker:
        msg = docker.group(2)
        severity = "ERROR" if re.search(r'error|fail|fatal|exception', msg, re.I) else \
                   "WARN"  if re.search(r'warn|warning',               msg, re.I) else "INFO"
        return NormalizedLog(
            timestamp=docker.group(1),
            severity=severity,
            service=None,
            message=msg,
            raw=raw_log, source_format="docker_log"
        )

    # ── 7. ERROR/WARN prefix without timestamp ───────────────────────────
    # ERROR: some message
    # [ERROR] some message
    # CRITICAL: some message
    prefix = re.match(
        r'^\[?(CRITICAL|ERROR|WARN(?:ING)?|INFO|DEBUG)\]?[:\s]+(.+)$',
        raw_log, re.IGNORECASE
    )
    if prefix:
        lvl = prefix.group(1).upper()
        lvl = "WARN" if lvl == "WARNING" else lvl
        return NormalizedLog(
            timestamp=None,
            severity=lvl,
            service=None,
            message=prefix.group(2),
            raw=raw_log, source_format="prefixed_plain"
        )

    # ── 8. Fallback — plain text, infer severity from keywords ───────────
    severity = "ERROR" if re.search(r'error|fail|fatal|exception|refused|timeout|crash', raw_log, re.I) else \
               "WARN"  if re.search(r'warn|warning|slow|high|degraded',                  raw_log, re.I) else "INFO"

    return NormalizedLog(
        timestamp=None,
        severity=severity,
        service=None,
        message=raw_log,
        raw=raw_log, source_format="unknown"
    )


def normalized_to_prompt_string(n: NormalizedLog) -> str:
    """Converts a NormalizedLog into a clean string for the LLM prompt."""
    return (
        f"Timestamp  : {n.timestamp or 'unknown'}\n"
        f"Severity   : {n.severity}\n"
        f"Service    : {n.service or 'unknown'}\n"
        f"Message    : {n.message}\n"
        f"Log format : {n.source_format}\n"
        f"Raw log    : {n.raw}"
    )

In [5]:
# CHROMA VECTOR DB SETUP
# Persistent — survives restarts, no re-embedding every run
import chromadb
from chromadb.utils.embedding_functions import EmbeddingFunction
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

CHROMA_PATH = "./chroma_db"   # local persistent storage


class BedrockEmbeddingFunction(EmbeddingFunction):
    """
    Wraps AWS Bedrock Titan Embeddings so ChromaDB can call it directly.
    ChromaDB expects embed_documents(texts) → List[List[float]]
    """
    def __call__(self, input: list[str]) -> list[list[float]]:
        embeddings = []
        for text in input:
            response = bedrock_runtime.invoke_model(
                modelId="amazon.titan-embed-text-v2:0",
                contentType="application/json",
                accept="application/json",
                body=json.dumps({"inputText": text[:8000]})  # Titan max input
            )
            result = json.loads(response["body"].read())
            embeddings.append(result["embedding"])
        return embeddings


def _content_hash(text: str) -> str:
    """Stable ID for a chunk so we never duplicate on re-load."""
    return hashlib.md5(text.encode()).hexdigest()


def build_vector_db(runbook_dir: str = "runbook", force_rebuild: bool = False):
    """
    Loads runbook markdown files → splits by headers → embeds → stores in ChromaDB.
    Skips re-embedding if chunks already exist (checks by content hash).
    Set force_rebuild=True to wipe and re-embed everything.
    """
    client     = chromadb.PersistentClient(path=CHROMA_PATH)
    embed_fn   = BedrockEmbeddingFunction()
    collection = client.get_or_create_collection(
        name="inframind_runbooks",
        embedding_function=embed_fn,
        metadata={"hnsw:space": "cosine"}
    )

    if force_rebuild:
        client.delete_collection("inframind_runbooks")
        collection = client.get_or_create_collection(
            name="inframind_runbooks",
            embedding_function=embed_fn,
            metadata={"hnsw:space": "cosine"}
        )

    # Load and split markdown runbooks
    loader = DirectoryLoader(runbook_dir, glob="*.md", loader_cls=TextLoader)
    docs   = loader.load()
    print(f"📚 Loaded {len(docs)} runbook files")

    headers_to_split_on = [("#", "H1"), ("##", "H2"), ("###", "H3")]
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    splits = []
    for doc in docs:
        splits.extend(splitter.split_text(doc.page_content))
    print(f"✂️  Split into {len(splits)} chunks")

    # Only embed chunks not already in the DB
    existing_ids = set(collection.get()["ids"])
    new_docs, new_ids, new_metas = [], [], []

    for chunk in splits:
        chunk_id = _content_hash(chunk.page_content)
        if chunk_id not in existing_ids:
            new_docs.append(chunk.page_content)
            new_ids.append(chunk_id)
            new_metas.append(chunk.metadata)

    if new_docs:
        # ChromaDB batch add (embed in batches of 10 to avoid Bedrock throttle)
        batch_size = 10
        for i in range(0, len(new_docs), batch_size):
            collection.add(
                documents=new_ids[i:i+batch_size],   # using IDs slot for retrieval
                ids=new_ids[i:i+batch_size],
                metadatas=new_metas[i:i+batch_size],
            )
            # Add actual text separately so we can retrieve it
            collection.upsert(
                ids=new_ids[i:i+batch_size],
                documents=new_docs[i:i+batch_size],
                metadatas=new_metas[i:i+batch_size],
            )
        print(f"✅ Embedded and stored {len(new_docs)} new chunks")
    else:
        print("⚡ All chunks already in ChromaDB — skipping re-embedding")

    return collection

In [6]:
def get_context(query: str, collection, k: int = 6, 
                max_distance: float = 0.45) -> str:
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    
    docs      = results["documents"][0]
    distances = results["distances"][0]
    
    # Only keep chunks below distance threshold
    filtered = [
        doc for doc, dist in zip(docs, distances)
        if dist < max_distance
    ]
    
    # Always keep at least 2 chunks even if all are far
    if len(filtered) < 2:
        filtered = docs[:2]
    
    print(f"   🎯 Retrieved {len(docs)} chunks, kept {len(filtered)} after distance filter")
    return "\n---\n".join(filtered)

In [7]:
def track_usage(response_body, model_id):
    i_tokens = response_body.get("prompt_token_count", 0)
    o_tokens = response_body.get("generation_token_count", 0)
    cost     = (i_tokens * PRICING[model_id]["input"]) + \
               (o_tokens * PRICING[model_id]["output"])
    mlflow.log_metric("tokens_in",  i_tokens)
    mlflow.log_metric("tokens_out", o_tokens)
    mlflow.log_metric("cost_usd",   cost)
    return cost

In [8]:
def extract_json(text: str) -> str:
    start = text.find("{")
    end   = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found in model output")
    return text[start:end + 1]

In [9]:
def call_llama(prompt: str, model: str, max_tokens: int = 512) -> str:
    response = bedrock_runtime.invoke_model(
        modelId=model,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt":      prompt,
            "max_gen_len": max_tokens,
            "temperature": 0.1,
            "top_p":       0.9,
        })
    )
    result = json.loads(response["body"].read())
    track_usage(result, model)
    return result["generation"]

In [10]:
def call_mistral(prompt: str, max_tokens: int = 512) -> str:
    """Mistral uses different format AND different response keys."""
    mistral_prompt = f"<s>[INST] {prompt} [/INST]"
    response = bedrock_runtime.invoke_model(
        modelId="mistral.mistral-7b-instruct-v0:2",
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt":      mistral_prompt,
            "max_tokens":  max_tokens,
            "temperature": 0.1,
        })
    )
    result = json.loads(response["body"].read())
    return result.get("outputs", [{}])[0].get("text", "")

In [11]:
def investigate_incident(log, context, model):

    prompt = f"""
    <|begin_of_text|>

    <|start_header_id|>system<|end_header_id|>
    You are a Site Reliability Engineer investigating an infrastructure incident.

    Analyze the log carefully.

    IMPORTANT RULES:
    - Only use facts visible in the log.
    - Do NOT invent causes.
    - Use the runbook context to identify known failure patterns.

    Return structured bullet points under these sections:

    Symptoms:
    - observable errors or behavior from the log

    Failing Component:
    - service or infrastructure element likely failing

    Infrastructure Layer:
    - Application / Database / Network / Kubernetes / Cloud / Unknown

    Possible Causes (based on log evidence and runbook):
    - list plausible technical causes

    <|eot_id|>

    <|start_header_id|>user<|end_header_id|>
    Runbook context:
    {context}

    Incident log:
    {log}

    <|eot_id|>

    <|start_header_id|>assistant<|end_header_id|>
    """

    return call_llama(prompt, model)

In [12]:
def infer_root_cause(investigation, context, model):

    prompt = f"""
<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>
You are a senior Site Reliability Engineer performing root cause analysis.

Use the RUNBOOK CONTEXT as the primary source of truth.

Rules:
- Root cause MUST be supported by the runbook.
- If the runbook suggests known failure modes, prioritize them.
- Do NOT invent causes not mentioned in the runbook.
- Use the investigation summary as evidence.

Explain reasoning step by step.

Output sections:

Root Cause Analysis:
- explanation

Evidence:
- log symptoms supporting the diagnosis

Runbook Reference:
- which runbook guidance supports this diagnosis

<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Investigation summary:
{investigation}

Runbook context:
{context}

<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

    return call_llama(prompt, model)

In [13]:
def generate_fix(reasoning, model):

    prompt = f"""
    <|begin_of_text|>

    <|start_header_id|>system<|end_header_id|>
    You are a Site Reliability Engineer writing remediation instructions.

    Provide concrete operational steps.

    Return three sections:

    Immediate Mitigation:
    - actions to restore service now

    Long-term Fix:
    - configuration or architecture fixes

    Verification Steps:
    - commands or checks to confirm the issue is resolved

    Do not leave any section empty.
    Do not use placeholders.

    <|eot_id|>

    <|start_header_id|>user<|end_header_id|>
    Root cause reasoning:
    {reasoning}
    <|eot_id|>

    <|start_header_id|>assistant<|end_header_id|>
    """
    return call_llama(prompt, model)

In [14]:
def format_rca(reasoning: str, fix: str, incident_id: str, model: str) -> RCAOutput:
    prompt = f"""<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>
Convert the RCA analysis into JSON.

Rules:
- Fill all fields with meaningful information.
- Use the remediation instructions for "immediate_fix".

Schema:
{json.dumps(json_schema)}
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Incident ID:
{incident_id}

Root cause reasoning:
{reasoning}

Remediation instructions:
{fix}
<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>"""

    result    = call_llama(prompt, model, max_tokens=1024)
    json_text = extract_json(result.strip())
    data      = json.loads(json_text)
    rca       = RCAOutput.model_validate(data)
    rca.incident_id = incident_id
    return rca

In [ ]:
# SENIOR SRE CRITIQUE  (Mistral-7B)
def senior_sre_critique(rca_report: RCAOutput, runbook_context: str) -> str:
    prompt = f"""You are a Staff SRE reviewing an RCA report for technical accuracy.
    Score the RCA from 0-10.
    Return output strictly as:

    SCORE: [X] | NOTE: explanation

    Runbook context:
    {runbook_context}

    RCA Report:
    {rca_report.model_dump_json()}"""

    critique = call_mistral(prompt, max_tokens=512)
    return critique

In [16]:
# DEEPEVAL  (Mistral-7B as judge)
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase


class BedrockMistralJudge(DeepEvalBaseLLM):
    def load_model(self):        return self
    def get_model_name(self):    return "Bedrock Mistral-7B Judge"

    def generate(self, prompt: str) -> str:
        return call_mistral(prompt, max_tokens=1024)

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)


bedrock_judge = BedrockMistralJudge()


def run_deepeval(log: str, context: str, rca: RCAOutput):
    try:
        test_case    = LLMTestCase(
            input=log,
            actual_output=rca.immediate_fix,
            retrieval_context=[context]
        )
        faith_metric = FaithfulnessMetric(threshold=0.7, model=bedrock_judge)
        rel_metric   = AnswerRelevancyMetric(threshold=0.7, model=bedrock_judge)
        faith_metric.measure(test_case)
        rel_metric.measure(test_case)
        return faith_metric.score, rel_metric.score
    except Exception as e:
        print(f"⚠️  DeepEval evaluation failed: {e}")
        return 0.0, 0.0

In [17]:
# MAIN RCA PIPELINE
def generate_infra_rca(
    raw_log:  str,
    context:  str = "",
    feedback: str = ""
) -> RCAOutput:

    incident_id = str(uuid.uuid4())

    # ── Normalize the log first ──────────────────────────────────────────
    normalized     = normalize_log(raw_log)
    structured_log = normalized_to_prompt_string(normalized)

    print(f"   📋 Log format detected: {normalized.source_format}")
    print(f"   🔍 Severity: {normalized.severity} | Service: {normalized.service or 'unknown'}")

    # ── Model selection ──────────────────────────────────────────────────
    selected_model = (
        "meta.llama3-70b-instruct-v1:0"
        if len(raw_log) > 2000
        else "meta.llama3-8b-instruct-v1:0"
    )

    # ── Agent chain ──────────────────────────────────────────────────────
    print("   🔎 Investigator Agent...")
    investigation = investigate_incident(structured_log, context, selected_model)

    if feedback:
        investigation += f"\n\nSenior SRE critique:\n{feedback}"

    print("   🧠 Root Cause Agent...")
    reasoning = infer_root_cause(investigation, context, selected_model)

    print("   🔧 Fix Agent...")
    fix = generate_fix(reasoning, selected_model)

    print("   📋 Formatter Agent...")
    rca = format_rca(reasoning, fix, incident_id, selected_model)

    rca.model_used = "Llama-3-70B" if "70b" in selected_model else "Llama-3-8B"
    return rca

In [18]:
# AUTONOMOUS WORKFLOW
def run_autonomous_workflow(
    log_text:    str,
    collection,               # ChromaDB collection
    max_retries: int = 2
) -> RCAOutput:

    if mlflow.active_run():
        mlflow.end_run()

    with mlflow.start_run(run_name=f"Incident_{datetime.now().strftime('%H%M%S')}"):
        mlflow.log_param("incident_log", log_text)

        # ── Normalize and log metadata ───────────────────────────────────
        normalized = normalize_log(log_text)
        mlflow.log_param("log_format",  normalized.source_format)
        mlflow.log_param("log_severity", normalized.severity)
        mlflow.log_param("log_service",  normalized.service or "unknown")

        # RAG 
        retrieval_query = (
            f"{normalized.severity} {normalized.message} "
            f"root cause diagnosis troubleshooting"
        )

        # Distance-filtered context
        knowledge = get_context(retrieval_query, collection, k=6, max_distance=0.45)

        mlflow.log_text(knowledge, "retrieved_context.txt")
        mlflow.log_param("retrieved_context_length", len(knowledge))

        current_feedback = ""

        for attempt in range(max_retries + 1):
            print(f"\n🚀 Execution Attempt {attempt + 1}...")

            rca = generate_infra_rca(log_text, context=knowledge, feedback=current_feedback)

            mlflow.log_metric(f"confidence_score_attempt_{attempt+1}", rca.confidence_score)
            mlflow.log_metric("rca_length", len(rca.summary))
            mlflow.log_dict({
                "incident_id":  rca.incident_id,
                "severity":     rca.severity,
                "summary":      rca.summary,
                "root_cause":   rca.root_cause,
                "immediate_fix": rca.immediate_fix,
            }, f"rca_output_attempt_{attempt+1}.json")

            # ── Senior SRE critique ──────────────────────────────────────
            critique = senior_sre_critique(rca, knowledge)
            print(f"\n--- RAW CRITIQUE RESPONSE ---\n{critique.strip()}")
            mlflow.log_text(critique, f"critique_attempt_{attempt+1}.txt")

            # ── DeepEval ─────────────────────────────────────────────────
            faith_score, rel_score = run_deepeval(log_text, knowledge, rca)
            mlflow.log_metric(f"faithfulness_score_attempt_{attempt+1}", faith_score)
            mlflow.log_metric(f"relevancy_score_attempt_{attempt+1}",    rel_score)

            # ── Parse critique score ─────────────────────────────────────
            match = re.search(r"score\s*:\s*\[?(\d+)\]?", critique, re.IGNORECASE)
            score = float(match.group(1)) / 10 if match else 0.0
            mlflow.log_metric(f"attempt_{attempt+1}_score", score)

            if score >= 0.8:
                print(f"✅ Success! Quality score {score} meets threshold.")
                mlflow.log_dict(rca.model_dump(), "final_rca_output.json")
                mlflow.log_text(critique, "final_senior_sre_review.txt")
                return rca
            else:
                print(f"⚠️  Score too low ({score}). Triggering self-correction loop...")
                current_feedback = critique

        print("🚨 Max retries reached. Returning best effort.")
        return rca

In [23]:
if __name__ == "__main__":

    # Build / load ChromaDB (skips re-embedding if already built)
    print("🗄️  Initialising ChromaDB...")
    collection = build_vector_db("runbook", force_rebuild=True)

    # ── Test with diverse log formats to verify normalizer ───────────────
    test_logs = [
        # Plain error string
        "ERROR 500: Database connection refused while connecting to host rds.cluster-inframind.internal",

        # Kubernetes syslog
        "E0115 10:23:45.123456   42 pod_workers.go:191] Error syncing pod: failed to pull image nginx:latest",

        # CloudWatch JSON
        '{"timestamp":"2024-01-15T10:23:45Z","level":"ERROR","service":"api-gateway","message":"connection refused to rds.cluster-inframind.internal:5432"}',

        # TLS timeout (plain)
        "ERROR: [Kubelet] Failed to pull image 'nginx:latest': RPC error: net/http: TLS handshake timeout",
    ]

    print("\n── Normalizer smoke test ──────────────────────────────────────")
    for log in test_logs:
        n = normalize_log(log)
        print(f"  [{n.source_format}] {n.severity} | {n.message[:80]}")

    # ── Run the full pipeline ────────────────────────────────────────────
    print("\n── Running full pipeline ──────────────────────────────────────")
    incident_log = test_logs[3]

    try:
        final_rca = run_autonomous_workflow(incident_log, collection)

        print("\n─── FINAL ANALYTICS REPORT ────────────────────────────────────")
        print("Incident ID  :", final_rca.incident_id)
        print("Severity     :", final_rca.severity)
        print("Summary      :", final_rca.summary)
        print("Root Cause   :", final_rca.root_cause)
        print("Immediate Fix:", final_rca.immediate_fix)
        print("Confidence   :", final_rca.confidence_score)
        print("Model        :", final_rca.model_used)

    except Exception as e:
        print(f"System Failure: {e}")
        raise

🗄️  Initialising ChromaDB...
📚 Loaded 11 runbook files
✂️  Split into 106 chunks


C:\Users\nasim_xhqpjmy\AppData\Local\Temp\ipykernel_2036\946549563.py:42: DeprecationWarning: The class BedrockEmbeddingFunction does not implement __init__. This will be required in a future version.
  embed_fn   = BedrockEmbeddingFunction()


✅ Embedded and stored 106 new chunks

── Normalizer smoke test ──────────────────────────────────────
  [prefixed_plain] ERROR | 500: Database connection refused while connecting to host rds.cluster-inframind.
  [k8s_syslog] ERROR | Error syncing pod: failed to pull image nginx:latest
  [cloudwatch_json] ERROR | connection refused to rds.cluster-inframind.internal:5432
  [prefixed_plain] ERROR | [Kubelet] Failed to pull image 'nginx:latest': RPC error: net/http: TLS handshak

── Running full pipeline ──────────────────────────────────────
   🎯 Retrieved 6 chunks, kept 2 after distance filter

🚀 Execution Attempt 1...
   📋 Log format detected: prefixed_plain
   🔍 Severity: ERROR | Service: unknown
   🔎 Investigator Agent...
   🧠 Root Cause Agent...
   🔧 Fix Agent...
   📋 Formatter Agent...

--- RAW CRITIQUE RESPONSE ---
SCORE: 8 | Note: The RCA report is clear and concise, and the root cause and immediate fix are correct. However, the diagnosis section could be improved by including the

✅ Success! Quality score 0.8 meets threshold.
🏃 View run Incident_183854 at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0/runs/bb2b7561e197454e807b400f55cccd6c
🧪 View experiment at: https://dagshub.com/nasim-raj-laskar/InfraMind.mlflow/#/experiments/0

─── FINAL ANALYTICS REPORT ────────────────────────────────────
Incident ID  : b639aeec-fdf4-4a82-89ab-183d89f792ff
Severity     : High
Summary      : Node security group missing outbound rule for port 443 causing TLS handshake timeout
Root Cause   : Node security group missing outbound rule for port 443
Immediate Fix: Add an outbound rule to the node's security group for port 443
Confidence   : 0.9
Model        : Llama-3-8B
